In [1]:
import torch
import time
import numpy as np
import matplotlib.pyplot as plt


from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

import sys
from pathlib import Path
# sys.path.insert(0, str(Path.cwd() / "src"))

from model_alignment_lab.utils.helpers import generate_response, format_example
from model_alignment_lab.evaluation.eval import log_parser, evaluate_to_dataframe

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

/home/derrjohn/git/model-alignment-lab/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


device(type='cpu')

In [2]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.11.0+cu130
13.0
False


In [ ]:
## Path Setup
root = Path.cwd()
output_dir = root.parent/"outputs"
datasets_dir = root.parent/"datasets"/"structured_json"
root

In [ ]:
print(root.is_dir())
print(output_dir.is_dir())
print(datasets_dir.is_dir())

# 1. Base Model Preparation

In [ ]:
MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32
)

model = model.to(device)
model.eval()

In [ ]:
prompt = """Return ONLY valid JSON.

Available tools:
- search_flights(origin, destination, departure_date, return_date)
- search_hotels(location, check_in_date, check_out_date)
- plan_trip(destination, start_date, end_date, budget_level)
- check_weather(location, date)

Use exact tool names and exact argument keys.

User request:
I want to go from San Diego to New York from 2026-05-10 to 2026-05-15 and keep it a medium budget."""

response = generate_response(model, tokenizer, prompt)
print(response)

In [ ]:
result = generate_response(
    model,
    tokenizer,
    prompt,
    max_new_tokens=30,
    benchmark=True
)
# print(result["text"])
# print()
print(f"Total Time: {np.round(result['total_time_s'],2)}")
print(f"Generated Tokens: {np.round(result['generated_tokens'],2)}")
print(f"Tokens per second: {np.round(result['tokens_per_second'],2)}")

In [ ]:
train_path = datasets_dir.joinpath("TRAIN_travel_tool_routing_100_samples_prompted_v2.jsonl")
test_path = datasets_dir.joinpath("TEST_travel_tool_routing_20_samples_prompted_v2.jsonl")
val_path = datasets_dir.joinpath("VAL_travel_tool_routing_20_samples_prompted_v2.jsonl")

In [ ]:
print(train_path.is_file())
print(test_path.is_file())
print(val_path.is_file())

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("json",
                             data_files={
                                 "train": str(train_path)
                             }
                            )["train"]
test_dataset = load_dataset("json",
                            data_files={
                                "test": str(test_path)
                            }
                           )["test"]
val_dataset = load_dataset("json",
                           data_files={
                               "val":str(val_path)
                           }
                          )["val"]

In [ ]:
train_dataset

In [ ]:
train_dataset["messages"][0]

In [ ]:
train_dataset = train_dataset.map(format_example, fn_kwargs={"tokenizer":tokenizer})
val_dataset = val_dataset.map(format_example, fn_kwargs={"tokenizer":tokenizer})
test_dataset = test_dataset.map(format_example, fn_kwargs={"tokenizer":tokenizer})

In [ ]:
train_dataset

In [ ]:
print(train_dataset["text"][0])

# 2. LoRA Finetune 

### LoRA Setup

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj"]
)


In [ ]:
model = get_peft_model(model, peft_config).to(model.device)
model.train()

In [ ]:
model.print_trainable_parameters()

## Training

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=str(output_dir/"smollm-tool-routing-lora"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    dataset_text_field="text",
    use_cpu=False,
    bf16=False,
    fp16=False,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
model

In [ ]:
final_path = str(output_dir/"smollm-tool-routing-lora")
trainer.model.save_pretrained(f"{final_path}/{ts}_final_adapter_v1")
tokenizer.save_pretrained(f"{final_path}/{ts}_final_adapter_v1")

## Model Evaluation

In [ ]:
trainer.state.log_history

In [ ]:
train_epochs, train_losses, eval_epochs, eval_losses = log_parser(trainer)

print(train_epochs)
print(train_losses)
print(eval_epochs)
print(eval_losses)

In [ ]:
figure, ax = plt.subplots()

ax.plot(train_epochs, train_losses, marker="o", label="Train Loss")
ax.plot(eval_epochs, eval_losses, marker="x", label="Eval Loss")

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training vs Evaluation Loss")

ax.legend()
ax.grid()

plt.show()

In [ ]:
prompt

In [ ]:
response = generate_response(model, tokenizer, prompt)
print(response)

In [ ]:
print(test_dataset["text"][0])

In [ ]:
df = evaluate_tool_calling_df(model, tokenizer, test_dataset)
df.head()

In [ ]:
df.tool_match.value_counts()

In [ ]:
df.arguments_match.value_counts()

In [ ]:
df.valid_json.value_counts()

## Train mas

In [ ]:
# Increase number of epochs
trainer.args.num_train_epochs = 6
trainer.train()

## Model v2 Eval

In [ ]:
train_epochs, train_losses, eval_epochs, eval_losses = log_parser(trainer)

print(train_epochs)
print(train_losses)
print(eval_epochs)
print(eval_losses)

In [ ]:
figure, ax = plt.subplots()

ax.plot(train_epochs, train_losses, marker="o", label="Train Loss")
ax.plot(eval_epochs, eval_losses, marker="x", label="Eval Loss")

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training vs Evaluation Loss")

ax.legend()
ax.grid()

plt.show()

In [ ]:
prompt 

In [ ]:
generate_response(model, tokenizer, prompt)

In [ ]:
df_2 = evaluate_tool_calling_df(model, tokenizer, test_dataset)
df_2.head()

In [ ]:
df_2.tool_match.value_counts()

In [ ]:
df_2.arguments_match.value_counts()

In [ ]:
df_2.valid_json.value_counts()

In [ ]:
trainer.model.save_pretrained(f"{final_path}/{ts}_final_adapter_v2")
tokenizer.save_pretrained(f"{final_path}/{ts}_final_adapter_v2")